## 0. Config Dir

In [ ]:
import os

if os.path.basename(os.getcwd()) in ["examples", "notebooks"]:
    os.chdir("..")

## Cell 1: Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import json
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time
from transformers import AutoModel, AutoImageProcessor

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Cell 2: Load Dataset Info

In [ ]:
JSON_PATH = "data/dataset/brain-tumor-mri-images-30-classes/DATA.json"
IMG_DIR = "data/dataset/brain-tumor-mri-images-30-classes"
OUTPUT_MODEL_CKPTS = "ckpts/dinov2_brain_tumor/best_dinov2_large_brain_tumor.pth"

os.makedirs(os.path.dirname(OUTPUT_MODEL_CKPTS), exist_ok=True)

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

dataset_info = []
classes_set = set()

for rel_path, info in data.items():
    safe_rel_path = rel_path.replace('\\', os.sep).replace('/', os.sep)
    full_img_path = os.path.join(IMG_DIR, safe_rel_path)
    if not os.path.exists(full_img_path):
        continue
    class_name = info['class']
    classes_set.add(class_name)
    dataset_info.append({'img_path': full_img_path, 'class': class_name})

classes_list = sorted(list(classes_set))
class_to_idx = {cls: idx for idx, cls in enumerate(classes_list)}
num_classes = len(classes_list)

print(f"Total images loaded: {len(dataset_info)}")
print(f"Number of classes identified: {num_classes}")

## Cell 3: Define PyTorch Dataset and Transforms

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, dataset_info, class_to_idx, transform=None):
        self.dataset_info = dataset_info
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataset_info)

    def __getitem__(self, idx):
        img_path = self.dataset_info[idx]['img_path']
        class_name = self.dataset_info[idx]['class']
        label = self.class_to_idx[class_name]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

IMG_SIZE = 518

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15)], p=0.4),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_info, val_info = train_test_split(
    dataset_info,
    test_size=0.2,
    random_state=42,
    stratify=[d['class'] for d in dataset_info]
)

train_dataset = BrainTumorDataset(train_info, class_to_idx, train_transform)
val_dataset = BrainTumorDataset(val_info, class_to_idx, val_transform)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## Cell 4: Load DINOv2 Large from Hugging Face

In [ ]:
print("Loading DINOv2-large from Hugging Face...")
backbone = AutoModel.from_pretrained("facebook/dinov2-large")
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-large")

for param in backbone.parameters():
    param.requires_grad = False

class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes, hidden_dim=768):
        super(DINOv2Classifier, self).__init__()
        self.backbone = backbone
        self.embed_dim = backbone.config.hidden_size  # 1024 for large
        self.classifier = nn.Sequential(
            nn.Linear(self.embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        outputs = self.backbone(x)
        features = outputs.last_hidden_state[:, 0]  # CLS token
        out = self.classifier(features)
        return out

model = DINOv2Classifier(backbone, num_classes).to(device)
print(f"Model initialized on {device}")
print(f"Embed dim: {model.embed_dim}, Num params: {sum(p.numel() for p in backbone.parameters()):,}")

## Cell 5: Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', 
                                                 factor=0.5, patience=2)

EPOCHS = 15
best_val_acc = 0.0

print("Starting training...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    start_time = time.time()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_acc = correct_train / total_train
    train_loss = train_loss / total_train

    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_acc = correct_val / total_val
    val_loss = val_loss / total_val

    scheduler.step(val_acc)

    epoch_time = time.time() - start_time
    print(f"Epoch [{epoch+1}/{EPOCHS}] ({epoch_time:.1f}s) - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), OUTPUT_MODEL_CKPTS)
        print(f"-> Best model saved with Val Acc: {best_val_acc:.4f}")

print("Training finished!")

## Cell 6: Evaluation and Visualization

In [ ]:
model.load_state_dict(torch.load(OUTPUT_MODEL_CKPTS, map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes_list))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes_list, yticklabels=classes_list)
plt.title('Confusion Matrix - DINOv2 Large')
plt.ylabel('Real Class')
plt.xlabel('Predicted Class')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix_dinov2_large.png')
plt.show()